# FFTW C2C F90 Sequential Sequana X

In [1]:
%%writefile  fc2csx.f90
program main
    use, intrinsic :: iso_c_binding
    implicit none
    include "fftw3.f03"
    integer, parameter :: N = 500, M = N, L = N
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    complex(C_DOUBLE_COMPLEX) :: s
    integer :: i, j, k
    double precision :: r, t1, t2
    
    call cpu_time(t1)    ! <--- time measurement
           
    ! in-place transform (note dimension reversal)
    cdata = fftw_alloc_complex(int(L * M * N, C_SIZE_T))
    call c_f_pointer(cdata, data, [L, M, N])

    ! create plan for in-place forward DFT (note dimension reversal)
    
    plan = fftw_plan_dft_3d(N, M, L, data, data, &
                            FFTW_FORWARD, FFTW_ESTIMATE)

    ! initialize data    
     do k = 1, N
        do j = 1, M
          do i = 1, L
            r = ((i-1)*M*L + (j-1)*M  + (k-1) + 1)*1E-6
            data(i, j, k) = cmplx(r, 0)
          enddo
        enddo
     enddo

    ! compute transform (as many times as desired)    
    call fftw_execute_dft(plan, data, data)
    s = sum(data)
    
    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
  
    call cpu_time(t2)    ! <--- time measurement
    
    ! show the result
    write(*, "('S: 'sf0.2spf0.2'j')", advance="no") s
    write(*, "('    T: 'sf0.4' s')") t2-t1  
end

Writing fc2csx.f90


In [2]:
%%bash
dir=/scratch/app/mathlibs/fftw/3.3.8_openmpi-3.1_gnu
gfortran -O3 -o fc2csx fc2csx.f90 \
    -L $dir/lib -l fftw3 -l m -I $dir/include

In [5]:
! hostname

sdumont18


In [6]:
! ls -lh fc2csx

-rwxr-xr-x 1 eduardo.miranda2 ampemi 14K Mar  5 17:00 fc2csx


In [7]:
! mpiexec -n 1 ./fc2csx

S: 125.00+.00j    T: 4.2533 s


In [8]:
! mpiexec -n 1 ./fc2csx

S: 125.00+.00j    T: 4.2947 s


In [7]:
! mpiexec -n 1 ./fc2csx

S: 125.00+.00j    T: 4.2533 s
